# Documents Validation

Use AI in SQL to validate PDF documents against a set of rules.

## The scenario

This demo checks customs paperwork for **Mexico-to-US auto parts shipments**. Each shipment is supported by six kinds of document: a commercial invoice, a packing list, a bill of lading (carta porte), a pedimento, a CBP 7501 entry summary, and a USMCA certification of origin. Missing fields, mismatched weights or values, and incomplete HTS codes can delay a shipment at the border, and finding them by hand across many PDFs is slow and error-prone.

## The approach: let each technology do what it is best at

| Step | Done by | Why |
|---|---|---|
| Read messy PDFs and pull out fields | **LLM agent** | Documents differ in layout and wording |
| Check the fields against the rules | **SQL** | Rules should be exact, repeatable, and auditable |
| Explain the findings and the fixes | **LLM agent** | Plain-language answers for the person filing |

The model is only trusted to *extract*, and it is told to return `null` rather than guess. The compliance checks themselves are ordinary SQL, so you can read and test them. The final agent can only report findings that come from a tool result.

## What you will do

1. Register a Claude model and point AIDB at a folder of 18 sample PDFs.
2. Build an **AI pipeline** that parses the PDFs into text.
3. Use an **extraction agent** to turn each document into structured fields.
4. Define **required-field rules** and a **findings view** of problems.
5. Wrap the findings in **SQL tools** and ask a **validator agent** which shipments are ready to file.

## Prerequisites

- A Postgres database with AIDB installed and permission to restart it.
- An Anthropic (Claude) API key.
- The 18 sample PDFs in `/var/lib/pgsql/pgfs/customs/`.

## Claude API Key

This demo creates agents that use a Claude Sonnet model, so an Anthropic API key is required. Run the next cell and paste your key when prompted. It is held in memory only and passed to the database when the model is registered.

In [ ]:
# Provide your Claude API Key
import json
from getpass import getpass
claude_api_key = getpass("Claude API key: ")
creds = json.dumps({"api_key": claude_api_key})

## Connect to the database

Creates a SQLAlchemy engine for the `aidb` database and attaches the `%sql` magics to it. `pool_pre_ping=True` checks connections before reuse, and named parameters are enabled so `%%sql` cells can use Python variables such as `:creds`.

In [ ]:
from sqlalchemy import create_engine
import os, pwd

PGUSER = pwd.getpwuid(os.geteuid()).pw_name
PGPORT="5432"
PGDATABASE = os.environ.get("PGDATABASE", "aidb")

if PGUSER == "postgres":
    PGSOCKET = "/run/postgresql"
else:
    PGSOCKET = "/tmp"

conn = create_engine(
    f"postgresql+psycopg2://{PGUSER}@/{PGDATABASE}?host={PGSOCKET}&port={PGPORT}"
)

%reload_ext sql
%sql conn
%config SqlMagic.named_parameters = "enabled"

## Register Claude Sonnet

`aidb.create_model` registers the Claude Sonnet 5 model inside the database under the name `claude_model_document_validation`. Both agents in this notebook (the extractor and the validator) use it. Any earlier registration with the same name is deleted first so the notebook can be re-run.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'claude_model_document_validation';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.models WHERE name = v_name) THEN
        PERFORM aidb.delete_model(v_name);
    END IF;

    PERFORM aidb.create_model(
        name        => v_name,
        provider    => 'anthropic_messages',
        config      => aidb.anthropic_messages_config(model => 'claude-sonnet-5'),
        credentials => CAST(:creds AS JSONB),
        replace_credentials => true
    );
END
$do$;

## Allowed Local Paths
You can manage how the PGFS interacts with your local data with the `pgfs.allowed_local_fs_paths` configuration setting.

By default, PGFS is only able to access files in `/tmp/pgfs` of the local filesystem. This setting is a security measure to prevent the extension from accessing files it shouldn't.

To allow PGFS to access files in other locations, add those locations to the `pgfs.allowed_local_fs_paths` GUC. This parameter accepts a colon-separated list of file system paths that PGFS is allowed to access.

Example:
`ALTER SYSTEM SET pgfs.allowed_local_fs_paths = '/tmp/pgfs:/tmp/other';`

*Note: After changing this setting, you must restart the database.*

First, let's get the current value.

In [ ]:
#load extension into memory by getting the version.
%sql SELECT pgfs.pgfs_version();

result = %sql show pgfs.allowed_local_fs_paths
path = result[0][0]

storage_location = f"file://{path}/" 
print(storage_location)

## Sample Validation PDF files
Claude was used to generate 18 PDF files for this demo. There are three shipments with six documents each. Some of the PDFs have incorrect or incomplete information on purpose, so there are problems for the validation to find.<br><br>
*Note: If the files aren't present, upload the files to allowed local filesystem path.*

In [ ]:
%%bash
mkdir -p $HOME/pgfs
rm -f $HOME/pgfs/validation
ln -s /tmp/pgfs/validation $HOME/pgfs/validation

## Create storage location

This example uses a local file location. The cell creates the folder on the database server where the PDFs will be stored.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'customs_pdfs_storage_location';
BEGIN
    IF EXISTS (SELECT NULL FROM pgfs.v_server_options WHERE srvname = v_name) THEN
        PERFORM pgfs.delete_storage_location(v_name);
    END IF;

    PERFORM pgfs.create_storage_location(v_name, :storage_location);
END
$do$;

## Create AIDB Volume

A **volume** presents the files in a storage location to AIDB as a source that pipelines can read. Here the volume `customs_pdfs_volume` covers the root of the storage location, and the `'Pdf'` format tells AIDB to treat the files as PDF documents. The Volume will be the source for an AI Pipeline.

The first statement deletes any volume previously created on this storage location so the cell can be re-run.

In [ ]:
%%sql
DO $do$
DECLARE
    v_schema text := 'public';
    v_name text := 'customs_pdfs_volume';
    v_fullname text := v_schema || '.' || v_name;
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.volumes WHERE schema = v_schema AND volume = v_name) THEN
        PERFORM aidb.delete_volume(v_fullname);
    END IF;

    PERFORM aidb.create_volume(v_name, 'customs_pdfs_storage_location', 'validation/', 'Pdf');
END
$do$;

## Create the AI Pipeline

AI Pipelines can ingest, prepare, and vectorize data for semantic retrieval and RAG applications. They integrate pgvector search capabilities and can keep the knowledge base synchronized with your source data automatically.

This pipeline has three steps:

1. **`ParsePdf`** extracts the text from each PDF. `Structured` parsing preserves layout such as tables, and `allow_partial_parsing` lets the pipeline continue if part of a file cannot be read. The parsed text is saved to a table called `customs_parsed_text` (`intermediate_destination`), with one row per document. **This table is what the extraction agent reads in the next steps.**
2. **`ChunkText`** splits the text into pieces of about 100 words (never more than 120), with 10 words of overlap between neighbouring chunks. This improves the accuracy of searches in large documents.
3. **`KnowledgeBase`** creates a vector embedding for each chunk with the `bge-small-en-v1.5-f16` model.

The validation in this demo works from the full parsed text, so the chunks and embeddings are not used again here. They make the same documents available for semantic search and RAG if you want to add that later.

`auto_processing => 'Disabled'` means the pipeline runs only when asked, and `aidb.run_pipeline` does that. This step may take a little while.

In [ ]:
%%sql

DO $do$
DECLARE
    v_name text := 'customs_pdfs';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.pipelines WHERE name = v_name) THEN
        PERFORM aidb.delete_pipeline(v_name);
    END IF;

    PERFORM aidb.create_pipeline(
        name           => v_name,
        source         => 'customs_pdfs_volume',
        step_1         => 'ParsePdf',
        step_1_options => aidb.pdf_parse_config(method => 'Structured', allow_partial_parsing => true)
                          || jsonb_build_object('intermediate_destination', jsonb_build_object('enabled', true, 'destination', 'customs_parsed_text')),
        step_2         => 'ChunkText',
        step_2_options => aidb.chunk_text_config(desired_length => 100, max_length => 120, overlap_length => 10, strategy => 'words')
                          || jsonb_build_object('intermediate_destination', jsonb_build_object('enabled', true)),
        step_3         => 'KnowledgeBase',
        step_3_options => aidb.knowledge_base_config(model => 'bge-small-en-v1.5-f16', data_format => 'Text'),
        auto_processing => 'Disabled'
    );
    
    PERFORM aidb.run_pipeline(v_name);
END
$do$;

## Create Agent

This agent extracts fields from the parsed PDF files. Read its `instructions` carefully, because they are what make the extraction trustworthy:

- **Never guess.** Any field that is absent or blank must be returned as `null`. The agent is told never to infer a value or copy it from somewhere else. A missing value should show up as missing so the rules can flag it.
- **Fixed vocabularies and formats.** `doc_type` must be one of six exact values, numbers contain digits and a decimal point only, and HTS codes are copied exactly as printed.
- **Document-specific definitions.** For example, `total_goods_value` means a different printed field on an invoice, a pedimento, and an entry summary.

`output_type` defines a structured response with 19 named, typed fields (`TEXT` or `FLOAT`), so the agent returns a predictable JSON object instead of free-form prose. That is what makes it possible to load the results straight into a table.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'customs_extractor';
BEGIN
    PERFORM aidb.delete_agent(v_name, force => true);

    PERFORM aidb.create_agent(
      name         => v_name,
      instructions => $$You extract fields from ONE customs document (commercial invoice, packing list, bill of lading / carta porte, pedimento, CBP 7501 entry summary, or USMCA certification of origin).
    Rules:
    - Return null for any field that is absent, blank, or not printed on the document. NEVER infer, guess, or copy a value from elsewhere.
    - doc_type must be exactly one of: invoice, packing_list, bol, pedimento, entry_summary, usmca_cert.
    - invoice_no is the commercial invoice number referenced on the document (e.g. INV-2026-10428). It is never a purchase order number.
    - po_no is the customer purchase order number. It has the form LETTERS-PO-DIGITS (e.g. TTAP-PO-5589).
    - Bill of lading only: the "PO / INVOICE" cell holds the PO number followed by the invoice number. Text extraction may join them with no space or line break, e.g. "ABCD-PO-1234INV-2026-0001". Split right after the digits that end the PO number: po_no = "ABCD-PO-1234", invoice_no = "INV-2026-0001". Likewise "WXYZ-PO-9876ZZ-2026-555" gives po_no "WXYZ-PO-9876" and invoice_no "ZZ-2026-555". If they are already separated, copy each as printed.
    - Numbers: digits and decimal point only, no commas or units.
    - total_goods_value: invoice = Subtotal (goods) excluding freight; pedimento = Valor Comercial USD; entry_summary = Total Entered Value. null for other doc types.
    - gross_weight_kg: total gross weight in kg. null if not shown.
    - hts_codes: every HTS / fraccion code printed, comma separated, exactly as printed (keep dots, do not pad or shorten).
    - line_qtys: for invoice and packing_list only, "PARTNO=QTY" pairs separated by semicolons, using the Part # on each line (use "various" if none).
    - origin_criteria: USMCA origin criterion letters (A-D) printed on the document, comma separated. null if none are printed.
    - signature_name / signature_date: only the name and date printed under the signature lines. null if those are blank.
    - has_vague_description: "yes" if any goods description is too generic to classify (e.g. "misc. parts"), else "no". null for doc types with no goods list.
    - missing_unit_price_lines: count of invoice line items with no unit price printed. 0 if none, null for other doc types.
    $$,
      model        => 'claude_model_document_validation', 
      output_type  => aidb.output_type(
          aidb.output_field('doc_type',                'TEXT'),
          aidb.output_field('invoice_no',              'TEXT'),
          aidb.output_field('po_no',                   'TEXT'),
          aidb.output_field('importer_name',           'TEXT'),
          aidb.output_field('importer_ein',            'TEXT'),
          aidb.output_field('currency',                'TEXT'),
          aidb.output_field('incoterm',                'TEXT'),
          aidb.output_field('seal_no',                 'TEXT'),
          aidb.output_field('hts_codes',               'TEXT'),
          aidb.output_field('line_qtys',               'TEXT'),
          aidb.output_field('total_goods_value',       'FLOAT'),
          aidb.output_field('gross_weight_kg',         'FLOAT'),
          aidb.output_field('origin_criteria',         'TEXT'),
          aidb.output_field('blanket_period_end',      'TEXT'),
          aidb.output_field('certifier_email',         'TEXT'),
          aidb.output_field('signature_name',          'TEXT'),
          aidb.output_field('signature_date',          'TEXT'),
          aidb.output_field('has_vague_description',   'TEXT'),
          aidb.output_field('missing_unit_price_lines','FLOAT')
      )
    );
END
$do$;

## Extract fields from documents

This uses the agent to extract the fields from the parsed PDF files.

The `INSERT ... SELECT` calls `aidb.agent_converse` once for every row of `customs_parsed_text` (that is, once per PDF), passing the file name and its text. For each document it stores:

- `source_id`: the file it came from.
- `fields`: the extracted values as `jsonb`, when the call succeeded.
- `error`: the error text, when it did not.

The final query is a quick check. As the comment says, expect 18 rows, six document types, and three distinct invoice numbers. Because this makes 18 model calls it can take a few minutes. If any row has an `error`, simply re-run the cell.

In [ ]:
%%sql
DO $do$
DECLARE
    v_rec record;
    v_fields jsonb;
    v_error text;
BEGIN
    DROP TABLE IF EXISTS customs_extracted CASCADE;
    CREATE TABLE customs_extracted (
      source_id text PRIMARY KEY,
      fields    jsonb,
      error     text
    );

    FOR v_rec IN SELECT s.source_id, s.value
        FROM customs_parsed_text s LOOP
        SELECT message, error 
        FROM aidb.agent_converse('customs_extractor',
                                 'Source file: ' || v_rec.source_id || E'\n\n' || v_rec.value)
        INTO v_fields, v_error;
        INSERT INTO customs_extracted (source_id, fields, error)
        VALUES (v_rec.source_id, v_fields, v_error);
    END LOOP;
END
$do$;


## Create relational view

The extracted fields are stored in JSON, so this step converts them to relational columns that SQL can filter and join.

- `customs_val` is a small cleaning function. It trims whitespace and turns empty strings and placeholders such as `null`, `n/a`, `none`, and `not provided` into a true SQL `NULL`. This protects the rules from the different ways a model might say "nothing here".
- `customs_docs` is a view with one row per document and one column per extracted field, with weights, values, and counts cast to numbers.

In [ ]:
%%sql
CREATE OR REPLACE FUNCTION customs_val(j jsonb, k text) RETURNS text
LANGUAGE sql IMMUTABLE AS $$
  SELECT CASE WHEN lower(btrim(coalesce(j->>k, ''))) IN ('', 'null', 'n/a', 'none', 'not provided')
              THEN NULL ELSE btrim(j->>k) END
$$;

CREATE OR REPLACE VIEW customs_docs AS
SELECT source_id,
       customs_val(fields,'doc_type')                      AS doc_type,
       customs_val(fields,'invoice_no')                    AS invoice_no,
       customs_val(fields,'po_no')                         AS po_no,
       customs_val(fields,'importer_name')                 AS importer_name,
       customs_val(fields,'importer_ein')                  AS importer_ein,
       customs_val(fields,'currency')                      AS currency,
       customs_val(fields,'incoterm')                      AS incoterm,
       customs_val(fields,'seal_no')                       AS seal_no,
       customs_val(fields,'hts_codes')                     AS hts_codes,
       customs_val(fields,'line_qtys')                     AS line_qtys,
       customs_val(fields,'total_goods_value')::numeric    AS total_goods_value,
       customs_val(fields,'gross_weight_kg')::numeric      AS gross_weight_kg,
       customs_val(fields,'origin_criteria')               AS origin_criteria,
       customs_val(fields,'blanket_period_end')            AS blanket_period_end,
       customs_val(fields,'certifier_email')               AS certifier_email,
       customs_val(fields,'signature_name')                AS signature_name,
       customs_val(fields,'signature_date')                AS signature_date,
       customs_val(fields,'has_vague_description')         AS has_vague_description,
       customs_val(fields,'missing_unit_price_lines')::numeric AS missing_unit_price_lines,
       fields
FROM customs_extracted
WHERE fields IS NOT NULL;

## Create rules

Each document type must contain certain elements. The `required_fields` table lists, for every document type, which fields are required and the regulation or reason behind each one (for example, the commercial invoice requirements in 19 CFR 141.86 and the USMCA certification data elements).

Keeping the rules in a table rather than in code makes them easy to review and change. To add a requirement, insert a row.

In [ ]:
%%sql

DROP TABLE IF EXISTS required_fields;
CREATE TABLE required_fields (doc_type text, field text, rule text);
INSERT INTO required_fields VALUES
 ('invoice','invoice_no','19 CFR 141.86'),
 ('invoice','po_no','19 CFR 141.86'),
 ('invoice','importer_name','19 CFR 141.86'),
 ('invoice','importer_ein','Importer of record number'),
 ('invoice','currency','19 CFR 141.86(a)(11): currency of sale'),
 ('invoice','incoterm','Terms of sale needed for value determination'),
 ('invoice','hts_codes','HTS classification'),
 ('invoice','total_goods_value','19 CFR 141.86'),
 ('invoice','gross_weight_kg','19 CFR 141.86(a)(9)'),
 ('invoice','origin_criteria','USMCA claim needs origin criterion'),
 ('invoice','signature_name','Signed declaration'),
 ('invoice','signature_date','Signed declaration'),
 ('packing_list','invoice_no','Reference to invoice'),
 ('packing_list','gross_weight_kg','Weights required'),
 ('packing_list','line_qtys','Quantities required'),
 ('bol','invoice_no','Reference to invoice'),
 ('bol','seal_no','Seal number required for truck entry'),
 ('bol','gross_weight_kg','Weights required'),
 ('pedimento','invoice_no','Reference to invoice'),
 ('pedimento','incoterm','Incoterm'),
 ('pedimento','total_goods_value','Declared commercial value'),
 ('pedimento','gross_weight_kg','Weights required'),
 ('entry_summary','invoice_no','Reference to invoice'),
 ('entry_summary','importer_ein','CBP 7501 box 8: importer of record number'),
 ('entry_summary','hts_codes','CBP 7501 HTS'),
 ('entry_summary','total_goods_value','CBP 7501 entered value'),
 ('usmca_cert','invoice_no','Reference to shipment'),
 ('usmca_cert','origin_criteria','USMCA Art. 5.2 data element 6'),
 ('usmca_cert','blanket_period_end','USMCA Art. 5.2 data element 7'),
 ('usmca_cert','certifier_email','USMCA Art. 5.2 data element 1'),
 ('usmca_cert','signature_date','USMCA Art. 5.2 data element 8');

## Findings

Compare the parsed fields with the rules. `customs_findings` is a view that combines six checks into one list of problems, each with the shipment, the document, a finding type, the detail, and the rule behind it:

| Finding type | What it catches |
|---|---|
| `MISSING_FIELD` | A required field is empty. The `found_elsewhere` column shows where the same value appears in another document of the same shipment, so the fix can be a copy-and-paste |
| `SHORT_HTS` | An HTS code on an invoice or entry summary has fewer than 10 digits |
| `MISSING_UNIT_PRICE` / `VAGUE_DESCRIPTION` | An invoice line has no unit price, or a goods description is too generic to classify |
| `WEIGHT_MISMATCH` | Gross weight differs across the documents of one shipment |
| `VALUE_MISMATCH` | Declared goods value differs across the invoice, pedimento, and entry summary |
| `QTY_MISMATCH` | A part's quantity differs between the invoice and the packing list |

The demo query at the end lists the findings. As the comment says, expect findings only for shipment `AB-2026-7731`. The other two shipments are clean.

In [ ]:
%%sql

CREATE OR REPLACE VIEW customs_findings AS
-- 5a. missing required fields (and where else in the same shipment the value exists)
SELECT d.invoice_no, d.doc_type, d.source_id,
       'MISSING_FIELD' AS finding_type,
       r.field AS detail,
       r.rule,
       (SELECT string_agg(DISTINCT o.doc_type || ': ' || customs_val(o.fields, r.field), '; ')
          FROM customs_docs o
         WHERE o.invoice_no = d.invoice_no AND o.source_id <> d.source_id
           AND customs_val(o.fields, r.field) IS NOT NULL) AS found_elsewhere
FROM customs_docs d
JOIN required_fields r ON r.doc_type = d.doc_type
WHERE customs_val(d.fields, r.field) IS NULL

UNION ALL
-- 5b. HTS codes shorter than 10 digits on invoice / entry summary
SELECT d.invoice_no, d.doc_type, d.source_id, 'SHORT_HTS', h.code,
       'U.S. HTS must be 10 digits for entry', NULL
FROM customs_docs d,
     LATERAL unnest(string_to_array(d.hts_codes, ',')) AS h(code)
WHERE d.doc_type IN ('invoice','entry_summary')
  AND length(regexp_replace(h.code, '\D', '', 'g')) < 10

UNION ALL
-- 5c. invoice line without unit price / vague description
SELECT invoice_no, doc_type, source_id,
       CASE WHEN missing_unit_price_lines > 0 THEN 'MISSING_UNIT_PRICE' ELSE 'VAGUE_DESCRIPTION' END,
       CASE WHEN missing_unit_price_lines > 0 THEN missing_unit_price_lines || ' line(s) with no unit price'
            ELSE 'Goods description too generic to classify' END,
       '19 CFR 141.86(a)(5): detailed description, unit price', NULL
FROM customs_docs
WHERE doc_type = 'invoice' AND (missing_unit_price_lines > 0 OR has_vague_description = 'yes')

UNION ALL
-- 5d. gross weight differs across documents of the same shipment
SELECT invoice_no, 'multiple', NULL, 'WEIGHT_MISMATCH',
       string_agg(doc_type || '=' || gross_weight_kg || ' kg', ', ' ORDER BY doc_type),
       'Weights must agree across documents', NULL
FROM customs_docs
WHERE gross_weight_kg IS NOT NULL AND invoice_no IS NOT NULL
GROUP BY invoice_no
HAVING count(DISTINCT gross_weight_kg) > 1

UNION ALL
-- 5e. goods value differs across invoice / pedimento / entry summary
SELECT invoice_no, 'multiple', NULL, 'VALUE_MISMATCH',
       string_agg(doc_type || '=' || total_goods_value, ', ' ORDER BY doc_type),
       'Declared value must agree across documents', NULL
FROM customs_docs
WHERE total_goods_value IS NOT NULL AND invoice_no IS NOT NULL
GROUP BY invoice_no
HAVING count(DISTINCT total_goods_value) > 1

UNION ALL
-- 5f. line quantity differs between invoice and packing list
SELECT i.invoice_no, 'multiple', NULL, 'QTY_MISMATCH',
       i.part_no || ': invoice=' || i.qty || ', packing_list=' || p.qty,
       'Invoice and packing list quantities must agree', NULL
FROM (SELECT invoice_no, split_part(x,'=',1) AS part_no,
             regexp_replace(split_part(x,'=',2), '[^0-9.]', '', 'g')::numeric AS qty
        FROM customs_docs, unnest(string_to_array(line_qtys, ';')) x
       WHERE doc_type = 'invoice' AND x LIKE '%=%') i
JOIN (SELECT invoice_no, split_part(x,'=',1) AS part_no,
             regexp_replace(split_part(x,'=',2), '[^0-9.]', '', 'g')::numeric AS qty
        FROM customs_docs, unnest(string_to_array(line_qtys, ';')) x
       WHERE doc_type = 'packing_list' AND x LIKE '%=%') p
  ON p.invoice_no = i.invoice_no AND btrim(p.part_no) = btrim(i.part_no)
WHERE i.qty <> p.qty;

-- Demo query: expect findings only for AB-2026-7731
SELECT invoice_no, doc_type, finding_type, detail, found_elsewhere
FROM customs_findings
ORDER BY invoice_no, finding_type, doc_type;


## SQL Tools

A **SQL tool** wraps a SQL statement so an agent can call it. Three tools are created, each reading from the views built above:

- `list_shipments`: every shipment with its number of documents and findings. It takes no parameters.
- `get_findings`: the validation findings for one shipment, looked up by invoice number.
- `get_shipment_docs`: the documents received for one shipment.

Each tool has a `description` that tells the agent when to use it, and `${invoice_no}` is a placeholder the agent fills in. Wrapping the checks as tools means the agent reads answers from the database instead of re-deriving them, so it cannot invent a finding.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'list_shipments';
BEGIN
    IF EXISTS (SELECT name FROM aidb.tools WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;
    PERFORM aidb.create_sql_tool(
      name          => v_name,
      description   => 'List every shipment (by invoice number) with its number of findings.',
      sql_statement => 'SELECT d.invoice_no, count(DISTINCT d.source_id) AS documents,
                               (SELECT count(*) FROM customs_findings f WHERE f.invoice_no = d.invoice_no) AS findings
                          FROM customs_docs d WHERE d.invoice_no IS NOT NULL GROUP BY d.invoice_no ORDER BY 1',
      params        => aidb.tool_params()
    );

    v_name := 'get_findings';
    IF EXISTS (SELECT name FROM aidb.tools WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;
    PERFORM aidb.create_sql_tool(
      name          => v_name,
      description   => 'Get validation findings (missing fields, mismatches, bad HTS codes) for one shipment by invoice number. found_elsewhere shows where a missing value appears in another document of the same shipment.',
      sql_statement => 'SELECT doc_type, finding_type, detail, rule, found_elsewhere
                          FROM customs_findings WHERE invoice_no = ${invoice_no} ORDER BY finding_type, doc_type',
      params        => aidb.tool_params(aidb.tool_param('invoice_no','TEXT','Commercial invoice number, e.g. AB-2026-7731'))
    );

    v_name := 'get_shipment_docs';
    IF EXISTS (SELECT name FROM aidb.tools WHERE name = v_name) THEN
        PERFORM aidb.delete_tool(v_name);
    END IF;
    PERFORM aidb.create_sql_tool(
      name          => v_name,
      description   => 'List the documents received for one shipment by invoice number.',
      sql_statement => 'SELECT source_id, doc_type FROM customs_docs WHERE invoice_no = ${invoice_no} ORDER BY source_id',
      params        => aidb.tool_params(aidb.tool_param('invoice_no','TEXT','Commercial invoice number'))
    );
END
$do$;


## Create Validator Agent

This second agent is the conversational front end for the person filing the shipment. It has the three tools above and instructions that say:

- Answer only from tool results (*"Never state a finding that did not come from a tool result"*).
- For each finding, explain what is wrong, which document it is in, and how to fix it. If `found_elsewhere` has a value, say it can be copied from that document.
- If a shipment has no findings, say it is ready to file.
- Be concise.

In [ ]:
%%sql
DO $do$
DECLARE
    v_name text := 'customs_validator';
BEGIN
    IF EXISTS (SELECT NULL FROM aidb.agents WHERE name = v_name) THEN
        PERFORM aidb.delete_agent(v_name, force => true);
    END IF;
    
    PERFORM aidb.create_agent(
      name         => 'customs_validator',
      instructions => $$You are a customs compliance assistant for Mexico-to-US auto parts shipments. Use list_shipments, get_shipment_docs and get_findings to answer. Never state a finding that did not come from a tool result. For each finding, say what is wrong, which document, and the fix; if found_elsewhere has a value, say it can be copied from that document. If a shipment has no findings, say it is ready to file. Be concise.$$,
      model        => 'claude_model_document_validation',
      tools        => ARRAY['list_shipments','get_shipment_docs','get_findings']
    );
END
$do$;

## Ask questions to the Validator Agent

Ask the agent in plain English. The first question gives an overview of all shipments. You should see that `AB-2026-7731` has problems and the other two shipments are ready to file. The second question asks for the specific problems with that shipment and how to fix them.

In [ ]:
from sqlalchemy import text
from IPython.display import Markdown, display
import pandas as pd

def ask(agent, question, conversation_id=None):
    """Ask the agent, render the answer, return (message, conversation_id).
    Pass a conversation_id from an earlier call to ask a follow-up."""
    with conn.begin() as c:
        df = pd.read_sql(
            text("""SELECT * FROM aidb.agent_converse(
                        CAST(:agent AS text), CAST(:q AS text),
                        conversation_id => CAST(:cid AS text))"""),
            c, params={"agent": agent, "q": question, "cid": conversation_id})
    row = df.iloc[0]
    if row["error"]:
        display(Markdown(f"**Agent error:** {row['error']}"))
    else:
        display(Markdown(row["message"]))
    # On an error the function returns a NULL id, so keep the one passed in
    new_id = row["conversation_id"]
    return row["message"], (conversation_id if pd.isna(new_id) else new_id)

AGENT = "customs_validator"

_, conversation_id = ask(AGENT, "Which shipments are ready to file and which are not?")

In [ ]:
_, conversation_id = ask(AGENT, "What is wrong with shipment AB-2026-7731 and how do I fix it?")

## Recap

You went from a folder of PDFs to a validation workflow inside Postgres:

1. A **pipeline** parsed the PDFs into text.
2. An **extraction agent** turned each document into structured fields, and was told to return `null` rather than guess.
3. **SQL rules** compared the fields with requirements and across documents.
4. A **validator agent** reported the findings and the fix for each one, using only tool results.

**Ideas to try next**

- Add a row to `required_fields` and see a new finding appear without changing any code.
- Edit or replace one of the PDFs, re-run the pipeline and extraction, and watch the findings change.
- Point the storage location at S3 instead of a local folder.
- Add a new document type and new checks to the findings view.